<a href="https://colab.research.google.com/github/nattagon256312-afk/RL-watering/blob/main/RL_Train_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install stable-baselines3[extra]
!pip install stable-baselines3[extra] tf2onnx tensorflow
!pip install onnxscript onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 9.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import random
import os
import torch
import tensorflow as tf

from gymnasium import Env
from gymnasium.spaces import Discrete, Box
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
class wateringEnv(Env):
  def __init__(self):
    self.action_space = Discrete(6)
    self.water_action = [0,20,40,60,80,100]
    self.observation_space = Box(low=-1.0, high=1.0, shape=(7,), dtype=np.float32)
    self.alpha = 0.5
    self.max_water = 800
    self.max_cap = 1000
    self.episode_days = 7
    self.max_steps = self.episode_days * 24

  def _get_obs(self):
    hour = self.time % 24
    time_remaining = self.time_length / self.max_steps

    return np.array([self.humidity / 1024,
                           self.water_total / self.max_cap,
                           self.max_water / self.max_cap,
                           time_remaining,
                           np.sin(2*np.pi*hour/24),
                           np.cos(2*np.pi*hour/24),
                           min(self.last_e/20, 1.0)], dtype=np.float32)


  def step(self, action):
    water = self.water_action[action]
    self.evap_state = 0.8*self.evap_state + random.gauss(0, 1.0)
    hour = self.time % 24
    daily = 1.0 + 0.5*np.cos(2*np.pi*(hour-12)/24)
    e_t = max(0.0, self.evap_scale * daily * (6.0 + self.evap_state))
    self.last_e = e_t

    water_loss= e_t*1
    actual_water = min(water, self.water_total)

    self.humidity = self.humidity + self.alpha*actual_water - water_loss
    self.humidity = np.clip(self.humidity, 0, 1024)
    self.water_total -= actual_water



    self.time_length -= 1
    self.time += 1

    low, high = 500, 750
    if self.humidity < low:
      reward = -(low - self.humidity) / 1024
    elif self.humidity > high:
      reward = -0.5 * (self.humidity - high) / 1024
    else:
      reward = 0.1

    reward -= 0.01 * (actual_water / 100)



    terminated = False
    truncated = self.time_length <= 0

    if self.water_total <= 0 and self.time_length > 0:
      reward -= 0.2

    return self._get_obs(), reward, terminated, truncated, {}

  def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.episode_days = random.choice([3, 5, 7, 10, 14])
    self.max_steps = self.episode_days * 24
    self.time_length = self.max_steps
    self.max_water = random.randint(300, 1000)
    self.water_total = self.max_water
    self.humidity = float(random.randint(0,1024))
    self.time = 0
    self.evap_scale = random.uniform(0.5, 2.0)
    self.evap_state = 0.0
    self.last_e = 0.0

    return self._get_obs(), {}
















In [ ]:
episodes = 5
env = wateringEnv()
for episode in range(1, episodes+1):
  state, info = env.reset()
  done = False
  score = 0

  while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    score += reward
  print (f'Episode:{episode} Score:{score}')

Episode:1 Score:-17.810830891570845
Episode:2 Score:-9.285582453017799
Episode:3 Score:-40.595928937158845
Episode:4 Score:-175.70565214040982
Episode:5 Score:-144.28739295215792


In [ ]:
log_path = "./ppo_watering_tensorboard/"
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)
model.learn(total_timesteps=1000000)

print("Training finished!")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./ppo_watering_tensorboard/PPO_1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
-----------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 186          |
|    ep_rew_mean          | -69.3        |
| time/                   |              |
|    fps                  | 1043         |
|    iterations           | 252          |
|    time_elapsed         | 494          |
|    total_timesteps      | 516096       |
| train/                  |              |
|    approx_kl            | 0.0039973212 |
|    clip_fraction        | 0.0659       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.341       |
|    explained_variance   | 0.991        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.601        |
|    n_updates            | 2510         |
|    policy_gradient_loss | -0.0049      |
|    value_loss           | 1.69         |
-----------------------------------------

In [ ]:
import torch
import torch.nn as nn

class ExtractPPOActor(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs):
        features = self.policy.features_extractor(obs)
        latent_pi, _ = self.policy.mlp_extractor(features)
        action_logits = self.policy.action_net(latent_pi)

        return action_logits


pure_actor = ExtractPPOActor(model.policy)
pure_actor.eval()


dummy_input = torch.randn(1, 7, dtype=torch.float32)

torch.onnx.export(
    pure_actor,
    dummy_input,
    "ppo_actor.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamo=False
)

print("Exported to ppo_actor.onnx successfully!")

Exported to ppo_actor.onnx successfully!


/tmp/ipykernel_2271/1152107144.py:30: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:218: DeprecationWarning: The feature will be removed. Please remove usage of this function
  setup_onnx_logging(verbose) as log_ctx,


In [ ]:

!pip install onnx2tf


!onnx2tf -i ppo_actor.onnx -o ppo_tf_output


!cp ppo_tf_output/ppo_actor_float32.tflite rl_model.tflite
!xxd -i rl_model.tflite > rl_model.h

print("Generated 'rl_model.h' successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 6              │ 6                │
│ Flatten    │ 1              │ 0                │
│ Gemm       │ 3              │ 3                │
│ Tanh       │ 2              │ 2                │
│ Model Size │ 21.3KiB        │ 21.4KiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 6              │ 6                │
│ Gemm       │ 3              │ 3                │
│ Tanh       │ 2              │ 2                │
│ Model Size │ 21.4KiB        │ 21.4KiB          │
└─